# **I. Airline Dataset Audit**

## **1. Introduction & Objectives**

Before diving into the File Format Benchmarking (Task 2), it is mandatory to audit our chosen dataset. This notebook evaluates the **2015 Flight Delays and Cancellations** dataset (`flights.csv` from Kaggle). 

The primary goal of this audit is to understand the raw data profile, verify the schema, and validate the existence of specific columns that will be utilized in subsequent partitioning and read query benchmarks.

**What will be audited:**
- Physical file size on disk.
- Total row and column counts.
- Schema and datatypes across the dataset.
- Missing values (Nulls) distribution.
- Validation of four critical columns required for Task 2:
  - `YEAR` and `MONTH` (Categorical columns for `.partitionBy()` testing).
  - `AIRLINE` and `ARRIVAL_DELAY` (Columns for the Read Query Benchmark: `SELECT AIRLINE, AVG(ARRIVAL_DELAY)`).
- Distinct cardinalities and distributions (to ensure `partitionBy` doesn't lead to a directory explosion).

**Scope limitations:**
This notebook is strictly for **Data Auditing**. It does **not** perform:
- Data cleaning or dropping rows.
- Filling missing values.
- Feature engineering or renaming columns.
- Format conversions or storage benchmarking (which will be handled in the main benchmark scripts).

In [1]:
# ============================================================
# AIRLINE DATASET AUDIT
# ============================================================

import os
from pyspark.sql import SparkSession, functions as F
from pyspark.storagelevel import StorageLevel

# ------------------------------------------------------------
# 1. Initialize Spark and read the raw dataset
# ------------------------------------------------------------
INPUT_FILE = r"C:\BigDataProject\data\raw\airline\flights.csv"

spark = (
    SparkSession.builder
    .appName("DatasetCandidateScreening")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

airline_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(INPUT_FILE)
    .persist(StorageLevel.MEMORY_AND_DISK)
)

# count() gets the row count and materializes the DataFrame so that
# subsequent audits do not have to re-read the entire CSV from scratch.
row_count = airline_df.count()
column_count = len(airline_df.columns)

# ------------------------------------------------------------
# 2. General information
# ------------------------------------------------------------
print("=" * 70)
print("BASIC DATASET INFORMATION")
print("=" * 70)

print(f"File size    : {os.path.getsize(INPUT_FILE) / (1024**2):,.2f} MB")
print(f"Row count    : {row_count:,}")
print(f"Column count : {column_count}")
print(f"Spark version: {spark.version}")

print("\nColumns:")
print(airline_df.columns)

print("\nSchema:")
airline_df.printSchema()

print("\nSample rows:")
airline_df.show(5, truncate=False)

# ------------------------------------------------------------
# 3. Missing values of the entire dataset
# ------------------------------------------------------------
null_counts = (
    airline_df.select([
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in airline_df.columns
    ])
    .first()
    .asDict()
)

null_summary = spark.createDataFrame([
    (
        column,
        int(null_counts[column]),
        round(null_counts[column] / row_count * 100, 4)
    )
    for column in airline_df.columns
], ["column", "null_count", "null_percentage"])

print("\n" + "=" * 70)
print("NULL VALUE SUMMARY")
print("=" * 70)

null_summary.orderBy(F.desc("null_percentage")).show(
    column_count,
    truncate=False
)

# ------------------------------------------------------------
# 4. Check critical columns for Task 2
# ------------------------------------------------------------
critical_cols = ["YEAR", "MONTH", "AIRLINE", "ARRIVAL_DELAY"]

print("\n" + "=" * 70)
print("CRITICAL COLUMN CHECK")
print("=" * 70)

for c in critical_cols:
    if c in airline_df.columns:
        print(f"[FOUND] {c:<15} -> {dict(airline_df.dtypes)[c]}")
    else:
        print(f"[MISSING] {c}")

# ------------------------------------------------------------
# 5. Statistics of critical columns
# ------------------------------------------------------------
critical_stats = airline_df.agg(
    F.min("YEAR").alias("min_year"),
    F.max("YEAR").alias("max_year"),
    F.countDistinct("YEAR").alias("distinct_years"),

    F.min("MONTH").alias("min_month"),
    F.max("MONTH").alias("max_month"),
    F.countDistinct("MONTH").alias("distinct_months"),

    F.countDistinct("AIRLINE").alias("distinct_airlines"),

    F.min("ARRIVAL_DELAY").alias("min_arrival_delay"),
    F.max("ARRIVAL_DELAY").alias("max_arrival_delay"),
    F.avg("ARRIVAL_DELAY").alias("avg_arrival_delay"),
    F.sum(F.col("ARRIVAL_DELAY").isNull().cast("int"))
     .alias("null_arrival_delay")
)

print("\n" + "=" * 70)
print("CRITICAL COLUMN STATISTICS")
print("=" * 70)

critical_stats.show(truncate=False)

# ------------------------------------------------------------
# 6. Distribution by Month
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("MONTH DISTRIBUTION")
print("=" * 70)

(
    airline_df.groupBy("MONTH")
    .count()
    .orderBy("MONTH")
    .show(20, truncate=False)
)

# ------------------------------------------------------------
# 7. Distribution by Airline
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("AIRLINE DISTRIBUTION")
print("=" * 70)

(
    airline_df.groupBy("AIRLINE")
    .count()
    .orderBy(F.desc("count"))
    .show(50, truncate=False)
)

# ------------------------------------------------------------
# 8. End of audit
# ------------------------------------------------------------
airline_df.unpersist()

BASIC DATASET INFORMATION
File size    : 564.96 MB
Row count    : 5,819,079
Column count : 31
Spark version: 3.4.1

Columns:
['YEAR', 'MONTH', 'DAY', 'DAY_OF_WEEK', 'AIRLINE', 'FLIGHT_NUMBER', 'TAIL_NUMBER', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT', 'SCHEDULED_DEPARTURE', 'DEPARTURE_TIME', 'DEPARTURE_DELAY', 'TAXI_OUT', 'WHEELS_OFF', 'SCHEDULED_TIME', 'ELAPSED_TIME', 'AIR_TIME', 'DISTANCE', 'WHEELS_ON', 'TAXI_IN', 'SCHEDULED_ARRIVAL', 'ARRIVAL_TIME', 'ARRIVAL_DELAY', 'DIVERTED', 'CANCELLED', 'CANCELLATION_REASON', 'AIR_SYSTEM_DELAY', 'SECURITY_DELAY', 'AIRLINE_DELAY', 'LATE_AIRCRAFT_DELAY', 'WEATHER_DELAY']

Schema:
root
 |-- YEAR: integer (nullable = true)
 |-- MONTH: integer (nullable = true)
 |-- DAY: integer (nullable = true)
 |-- DAY_OF_WEEK: integer (nullable = true)
 |-- AIRLINE: string (nullable = true)
 |-- FLIGHT_NUMBER: integer (nullable = true)
 |-- TAIL_NUMBER: string (nullable = true)
 |-- ORIGIN_AIRPORT: string (nullable = true)
 |-- DESTINATION_AIRPORT: string (nullable 

DataFrame[YEAR: int, MONTH: int, DAY: int, DAY_OF_WEEK: int, AIRLINE: string, FLIGHT_NUMBER: int, TAIL_NUMBER: string, ORIGIN_AIRPORT: string, DESTINATION_AIRPORT: string, SCHEDULED_DEPARTURE: int, DEPARTURE_TIME: int, DEPARTURE_DELAY: int, TAXI_OUT: int, WHEELS_OFF: int, SCHEDULED_TIME: int, ELAPSED_TIME: int, AIR_TIME: int, DISTANCE: int, WHEELS_ON: int, TAXI_IN: int, SCHEDULED_ARRIVAL: int, ARRIVAL_TIME: int, ARRIVAL_DELAY: int, DIVERTED: int, CANCELLED: int, CANCELLATION_REASON: string, AIR_SYSTEM_DELAY: int, SECURITY_DELAY: int, AIRLINE_DELAY: int, LATE_AIRCRAFT_DELAY: int, WEATHER_DELAY: int]

## **Analysis of Results**

### **1. Dataset Scale**

Based on the general information output, we can record the following dimensions:

- **File size:** 564.96 MB
- **Row count:** 5,819,079
- **Column count:** 31

**Objective:** This confirms that the dataset size (~5.8 million rows, >500MB) is substantial enough to yield observable and meaningful differences in I/O performance during the format storage and read execution benchmarks. It also verifies that the loaded data aligns perfectly with the expected scale of the Kaggle 2015 Flight Delays dataset.

### **2. Schema Validation**

We verified the entire schema inferred by Spark from the `flights.csv` file. Specifically, we confirmed the presence and datatypes of the four critical columns required for Task 2:

| Column | Role in Task 2 | Audit Result |
|---|---|---|
| `YEAR` | Partition column | Validated (`integer`) |
| `MONTH` | Partition column | Validated (`integer`) |
| `AIRLINE` | Categorical column / `GROUP BY` | Validated (`string`) |
| `ARRIVAL_DELAY` | Numeric measure / `AVG()` | Validated (`integer`) |

Since all four critical columns exist with the correct datatypes, this dataset can be mapped almost directly to the canonical schema required by the project:

- `AIRLINE` → `Carrier`
- `ARRIVAL_DELAY` → `ArrDelay`
- `YEAR` → `Year`
- `MONTH` → `Month`

*(Note: Column renaming is not performed during this audit step; it will be handled in the benchmarking pipeline if this dataset is selected).*


### **3. Missing Values Summary**

Based on the `NULL VALUE SUMMARY` table, we identified the following data quality conditions:

- **High Missing Rates:** Columns related to specific delay reasons (e.g., `CANCELLATION_REASON`, `SECURITY_DELAY`, `WEATHER_DELAY`) have extreme null percentages ranging from 81% to over 98%.
- **Partitioning & Grouping Columns:** `YEAR`, `MONTH`, and `AIRLINE` have **0 missing values**, guaranteeing clean and reliable aggregations/partitioning.
- **Metric Column:** `ARRIVAL_DELAY` contains **105,071 missing values** (approx. 1.81%). These likely correspond to cancelled or diverted flights.

At this stage, we are purely observing the data state. **No preprocessing decisions** have been made yet regarding:
- Dropping nulls.
- Filling nulls.
- Filtering out cancelled flights.
- Filtering out diverted flights.

These preprocessing decisions will be evaluated after comparing this dataset with the NYC Taxi dataset.


### **4. Year and Month Distribution**

Checking the bounds and cardinality of the temporal columns:

- `min_year`: 2015
- `max_year`: 2015
- `distinct_years`: 1
- `min_month`: 1
- `max_month`: 12
- `distinct_months`: 12

**Conclusion:** The dataset natively covers exactly one year (2015) and all 12 months (1 to 12). Because these columns are already isolated and have low cardinality, they can be utilized directly for the partition experiment:
`.partitionBy("Year", "Month")` 
This can be done natively without the CPU overhead of deriving them from a timestamp column. Furthermore, this guarantees exactly 12 partition folders, safely avoiding the "Small File Problem".


### **5. Airline Distribution**

Checking the categorical grouping column:

- `distinct_airlines`: 14
- The `AIRLINE DISTRIBUTION` table shows a healthy spread across 14 carriers, with "WN" (Southwest) leading at ~1.26M rows and "VX" (Virgin America) at the tail with ~61k rows.

**Conclusion:** `AIRLINE` is a perfectly suited categorical variable with a highly manageable number of categories. This makes it ideal for the benchmark's aggregate query, mapping perfectly to the `GROUP BY Carrier` requirement.


### **6. Arrival Delay Profile**

Checking the numeric metric column:

- **Minimum arrival delay:** -87 (Early arrivals)
- **Maximum arrival delay:** 1971 (Extreme delays)
- **Average arrival delay:** ~4.41
- **Null values:** 105,071

**Conclusion:** `ARRIVAL_DELAY` behaves exactly as expected for a numeric column. Despite the ~1.8% null rate, it provides a wide range of valid integer values, making it highly suitable for the `AVG(ArrDelay)` computation required in the read query benchmark once the schema is standardized.


### **7. Conclusion for Candidate A**

Following this audit, we can summarize Candidate A (Airline 2015) against our project criteria:

| Evaluation Criteria | Airline 2015 Dataset |
|---|---|
| Spark successfully reads the dataset | Yes |
| Suitable categorical column exists | Yes (`AIRLINE`, 14 distinct) |
| Suitable numeric column exists | Yes (`ARRIVAL_DELAY`) |
| Native `Year` column available | Yes |
| Native `Month` column available | Yes |
| Needs Year/Month derivation logic | No (Ready to use) |
| Fits project's benchmark query | Yes |
| Notable missing values issues | Minor (1.8% in `ARRIVAL_DELAY`, acceptable) |

**Final Note:** We are not concluding which dataset is better at this stage. Candidate A will only be evaluated and compared against Candidate B (NYC Yellow Taxi 2024) once a similar data audit is completed for the second dataset.


# **I. NYC Yellow Taxi 2024: Download & Dataset Audit**

## **1. Introduction & Objectives**

This notebook evaluates **Candidate B: NYC TLC Yellow Taxi Trip Records 2024**. 
Unlike the Airline dataset which is a single file, NYC TLC publishes its data monthly. This candidate is locked to the following specifications:

- **Dataset:** Yellow Taxi Trip Records (Provider: NYC TLC)
- **Coverage:** January – December 2024 (12 files)
- **Native Format:** Parquet

**What this notebook executes:**
1. **Source Validation:** Verify that all 12 locally downloaded monthly files are available.
2. **Consolidation:** Verify schema consistency across all files and load them into a unified Spark DataFrame.
3. **Data Audit:** Inspect dataset dimensions (rows/columns), schema, missing values, and time range.
4. **Task 2 Compatibility Check:** Identify potential categorical/numeric fields for the query benchmark and assess the feasibility of extracting `Year` and `Month` from pickup timestamps.

**Scope Limitations:**
This notebook is strictly for **dataset validation and auditing**. It does **not** perform data cleaning, missing value imputation, column renaming, format conversion, or the actual benchmarking.

In [3]:
# ============================================================
# NYC YELLOW TAXI 2024 DATASET AUDIT
# ============================================================

# ------------------------------------------------------------
# 1. Define dataset location and expected monthly files
# ------------------------------------------------------------

NYC_DATA_DIR = r"C:\BigDataProject\data\raw\nyc_taxi"

expected_files = [
    f"yellow_tripdata_2024-{month:02d}.parquet"
    for month in range(1, 13)
]

file_paths = [
    os.path.join(NYC_DATA_DIR, filename)
    for filename in expected_files
]

# ------------------------------------------------------------
# 2. Verify that all 12 monthly files exist
# ------------------------------------------------------------

print("=" * 70)
print("SOURCE FILE CHECK")
print("=" * 70)

missing_files = []

for path in file_paths:
    filename = os.path.basename(path)

    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024 ** 2)
        print(f"[FOUND]   {filename:<35} {size_mb:>10.2f} MB")
    else:
        print(f"[MISSING] {filename}")
        missing_files.append(filename)

if missing_files:
    raise FileNotFoundError(
        f"Missing files: {missing_files}"
    )

print("\nAll 12 monthly files are available.")

# ------------------------------------------------------------
# 3. Start Spark
# ------------------------------------------------------------

spark = (
    SparkSession.builder
    .appName("NYCYellowTaxi2024Audit")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

# ------------------------------------------------------------
# 4. Check schema consistency across all 12 monthly files
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SCHEMA CONSISTENCY CHECK")
print("=" * 70)

taxi_schemas = {}

for month, path in enumerate(file_paths, start=1):
    taxi_schemas[month] = spark.read.parquet(path).schema

taxi_reference_schema = taxi_schemas[1]

for month in range(1, 13):
    status = (
        "MATCH"
        if taxi_schemas[month] == taxi_reference_schema
        else "DIFFERENT"
    )

    print(f"2024-{month:02d}: {status}")

taxi_all_schema_match = all(
    schema == taxi_reference_schema
    for schema in taxi_schemas.values()
)

print(f"\nAll 12 taxi_schemas identical: {taxi_all_schema_match}")

# ------------------------------------------------------------
# 5. Read all 12 monthly files into one DataFrame
# ------------------------------------------------------------

taxi_df = spark.read.parquet(*file_paths)

print("\n" + "=" * 70)
print("BASIC DATASET INFORMATION")
print("=" * 70)

row_count = taxi_df.count()
column_count = len(taxi_df.columns)

print(f"Row count   : {row_count:,}")
print(f"Column count: {column_count}")

print("\nColumns:")
print(taxi_df.columns)

print("\nSchema:")
taxi_df.printSchema()

print("\nSample rows:")
taxi_df.show(5, truncate=False)

# ------------------------------------------------------------
# 6. Calculate missing values for all columns
# ------------------------------------------------------------

null_counts = (
    taxi_df.select([
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in taxi_df.columns
    ])
    .first()
    .asDict()
)

null_summary = spark.createDataFrame([
    (
        column,
        int(null_counts[column]),
        round(null_counts[column] / row_count * 100, 4)
    )
    for column in taxi_df.columns
], ["column", "null_count", "null_percentage"])

print("\n" + "=" * 70)
print("NULL VALUE SUMMARY")
print("=" * 70)

null_summary.orderBy(
    F.desc("null_percentage")
).show(
    column_count,
    truncate=False
)

# ------------------------------------------------------------
# 7. Check important columns for this candidate
# ------------------------------------------------------------

important_cols = [
    "VendorID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "trip_distance",
    "total_amount"
]

print("\n" + "=" * 70)
print("IMPORTANT COLUMN CHECK")
print("=" * 70)

dtype_map = dict(taxi_df.dtypes)

for column in important_cols:
    if column in taxi_df.columns:
        print(f"[FOUND]   {column:<25} -> {dtype_map[column]}")
    else:
        print(f"[MISSING] {column}")

# ------------------------------------------------------------
# 8. Inspect important numeric and temporal fields
# ------------------------------------------------------------

summary = taxi_df.agg(
    F.min("tpep_pickup_datetime").alias("min_pickup"),
    F.max("tpep_pickup_datetime").alias("max_pickup"),

    F.countDistinct("VendorID").alias("distinct_vendors"),

    F.min("trip_distance").alias("min_trip_distance"),
    F.max("trip_distance").alias("max_trip_distance"),
    F.avg("trip_distance").alias("avg_trip_distance"),

    F.min("total_amount").alias("min_total_amount"),
    F.max("total_amount").alias("max_total_amount"),
    F.avg("total_amount").alias("avg_total_amount")
)

print("\n" + "=" * 70)
print("IMPORTANT FIELD STATISTICS")
print("=" * 70)

summary.show(truncate=False)

# ------------------------------------------------------------
# 9. Check whether YEAR and MONTH exist natively
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("YEAR / MONTH COMPATIBILITY")
print("=" * 70)

print(f"Native YEAR column : {'YES' if 'YEAR' in taxi_df.columns else 'NO'}")
print(f"Native MONTH column: {'YES' if 'MONTH' in taxi_df.columns else 'NO'}")

# ------------------------------------------------------------
# 10. Derive Year and Month temporarily for audit only
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("DERIVED YEAR / MONTH DISTRIBUTION")
print("=" * 70)

(
    taxi_df.groupBy(
        F.year("tpep_pickup_datetime").alias("YEAR"),
        F.month("tpep_pickup_datetime").alias("MONTH")
    )
    .count()
    .orderBy("YEAR", "MONTH")
    .show(50, truncate=False)
)

# ------------------------------------------------------------
# 11. Inspect VendorID distribution
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("VENDOR DISTRIBUTION")
print("=" * 70)

(
    taxi_df.groupBy("VendorID")
    .count()
    .orderBy(F.desc("count"))
    .show(20, truncate=False)
)

# ------------------------------------------------------------
# 12. Stop Spark
# ------------------------------------------------------------

spark.stop()

SOURCE FILE CHECK
[FOUND]   yellow_tripdata_2024-01.parquet          47.65 MB
[FOUND]   yellow_tripdata_2024-02.parquet          48.02 MB
[FOUND]   yellow_tripdata_2024-03.parquet          57.30 MB
[FOUND]   yellow_tripdata_2024-04.parquet          56.39 MB
[FOUND]   yellow_tripdata_2024-05.parquet          59.66 MB
[FOUND]   yellow_tripdata_2024-06.parquet          57.09 MB
[FOUND]   yellow_tripdata_2024-07.parquet          49.88 MB
[FOUND]   yellow_tripdata_2024-08.parquet          48.70 MB
[FOUND]   yellow_tripdata_2024-09.parquet          58.34 MB
[FOUND]   yellow_tripdata_2024-10.parquet          61.37 MB
[FOUND]   yellow_tripdata_2024-11.parquet          57.85 MB
[FOUND]   yellow_tripdata_2024-12.parquet          58.67 MB

All 12 monthly files are available.

SCHEMA CONSISTENCY CHECK
2024-01: MATCH
2024-02: MATCH
2024-03: MATCH
2024-04: MATCH
2024-05: MATCH
2024-06: MATCH
2024-07: MATCH
2024-08: MATCH
2024-09: MATCH
2024-10: MATCH
2024-11: MATCH
2024-12: MATCH

All 12 taxi_schema

## **Analysis of Results**

### **1. Source File Check**

- All 12 monthly files (January to December 2024) were successfully located. 
- The output clearly states `[FOUND]` for every month, confirming that Candidate B is correctly locked at the local snapshot level. 
- **Total files:** 12.
- *(Note: Individual file sizes range from ~47 MB to ~61 MB, summing to over 650 MB of raw compressed Parquet data).*

### **2. Schema Consistency**

- **Status:** `All 12 schemas identical: True`.
- Every monthly file from `2024-01` to `2024-12` returned a `MATCH` against the reference schema.
- **Conclusion:** The schema is perfectly stable across the entire year, allowing us to safely read all 12 files into a single, unified Spark DataFrame without facing schema evolution conflicts.

### **3. Dataset Size**

- **Row count:** 41,169,720
- **Column count:** 19
- **Conclusion:** With over 41 million rows, Candidate B is significantly larger than the Airline dataset (~5.8 million rows). This massive scale is excellent for stress-testing and highlighting performance differences in our I/O benchmarking (Task 2).

### **4. Native Schema**

We verified the presence of key fields:
- `VendorID` (integer)
- `tpep_pickup_datetime` / `tpep_dropoff_datetime` (timestamp)
- `trip_distance` / `total_amount` (double)

While this dataset contains natural categorical and numeric fields, **it lacks direct semantic equivalents** to the Airline dataset's `Carrier` and `ArrDelay`. If NYC Taxi is chosen, the benchmark query must be redefined according to taxi semantics rather than forcing Airline column names.

### **5. Missing Values**

Based on the `NULL VALUE SUMMARY`:
- **Critical Columns:** `VendorID`, `tpep_pickup_datetime`, `trip_distance`, and `total_amount` have **0 missing values (100% complete)**.
- Secondary columns (e.g., `passenger_count`, `RatecodeID`, `congestion_surcharge`) have exactly 4,091,232 nulls (~9.94%).
- At this stage, we are only observing. No dropping or filling of nulls is performed.

### **6. Time Coverage**

- **min_pickup:** `2002-12-31 16:46:07`
- **max_pickup:** `2026-06-26 23:53:12`
- **Conclusion:** Although this is the "2024" dataset, the presence of timestamps from 2002 and 2026 indicates **dirty data / data-quality issues**. These out-of-bounds records must be handled later if we want strict 2024 partitioning.

### **7. Year and Month Availability**
**
- **Native YEAR column:** NO
- **Native MONTH column:** NO
- **Conclusion:** Unlike the Airline dataset, NYC Taxi does not have native `Year` and `Month` columns. They must be derived from the `tpep_pickup_datetime` timestamp. This adds a CPU-bound preprocessing step (transformation) before we can execute `.partitionBy("Year", "Month")`.

### **8. Derived Year/Month Distribution**

The distribution table confirms that **over 99.9%** of the data properly falls within 2024 (Months 1 through 12, ranging from ~2.9M to ~3.8M rows per month). 
However, outlier years (2002, 2008, 2009, 2023, 2025, 2026) contain trace amounts of records (e.g., 1 to 18 rows). If we partition by Year/Month without filtering these out, Spark will create tiny, nearly empty partition directories, which directly contributes to the **Small File Problem**.

### **9. VendorID as a Categorical Variable**

- `VendorID` can technically be used for a categorical aggregate query (`GROUP BY VendorID`).
- However, the distribution is highly skewed: Vendor 2 dominates with ~31.4M rows, Vendor 1 has ~9.7M, and Vendors 6 & 7 have negligible counts. With only 4 distinct values, it is far less diverse than the 14 distinct carriers in the Airline dataset.

### **10. Numeric Measures**

Natural numeric metrics available are `trip_distance` and `total_amount`. Either can be used for aggregate queries like `AVG(trip_distance)`. 
*Note:* The statistics reveal extreme outliers (e.g., negative amounts of -2265, maximum distance of 398,608). Furthermore, there is no native "Delay" equivalent metric.

### **11. Candidate B Summary**

Following this audit, here is the final assessment of NYC Yellow Taxi 2024:

| Criteria | NYC Yellow Taxi 2024 |
|---|---|
| Full 12 monthly files present | Yes |
| 12 schemas identical | Yes |
| Row count | 41,169,720 |
| Column count | 19 |
| Has suitable categorical variable | Yes (`VendorID`), but heavily skewed (4 distinct) |
| Has suitable numeric variable | Yes (`trip_distance`, `total_amount`) |
| Has native Year | **No** |
| Has native Month | **No** |
| Can derive Year/Month | Yes (from `tpep_pickup_datetime`) |
| Natural equivalent to `Carrier` | No (requires semantic shift to `VendorID`) |
| Natural equivalent to `ArrDelay` | No (requires semantic shift to distance/amount) |
| Notable data-quality issues | Out-of-bounds dates (2002, 2026), negative amounts |
| Preprocessing required | **High** (Needs timestamp derivation & outlier filtering to avoid tiny partition folders) |

# **II. Dataset Comparison & Final Selection**

## **1. Objective**

This step evaluates which candidate dataset better satisfies the specific requirements of **Task 2 (File Format Benchmarking & Partition Management)**. Rather than declaring a universally "better" dataset, we assess them based on:
- Alignment with the required read query: `SELECT carrier, AVG(arr_delay)`
- Native support for the `.partitionBy("Year", "Month")` experiment.
- Minimal preprocessing overhead.
- Practicality for reproducible, multi-run local execution (warm-ups and iterations) to seamlessly integrate with ID4's workflow.

## **2. Candidate Summary**

| Criterion | Candidate A: Airline 2015 | Candidate B: NYC Yellow Taxi 2024 |
|---|---|---|
| **Source Structure** | 1 main CSV file | 12 monthly Parquet files |
| **Scale (Rows / Cols)** | 5,819,079 / 31 | 41,169,720 / 19 |
| **Native Categorical Field** | `AIRLINE` (14 distinct) | `VendorID` (4 distinct, heavily skewed) |
| **Native Numeric Measure** | `ARRIVAL_DELAY` | `trip_distance`, `total_amount` |
| **Native Year/Month Columns**| **Yes** | **No** (Derivation required) |
| **Semantic Match to Query** | **Exact match** (`Carrier`, `ArrDelay`) | **Requires query redesign** |
| **Partitioning Readiness** | Ready | Requires outlier filtering |
| **Preprocessing Required** | Low (Minor nulls in delay) | High (Out-of-bounds dates, negative amounts) |

## **3. Comparison by Benchmark Requirements**

### **3.1 Read Query Compatibility**
The assignment suggests benchmarking a query like `SELECT carrier, AVG(arr_delay)`. 
- **Airline:** Maps natively without semantic changes (`AIRLINE` → `Carrier`, `ARRIVAL_DELAY` → `ArrDelay`).
- **Taxi:** Would require substituting the concepts entirely (e.g., `GROUP BY VendorID` and `AVG(trip_distance)`). 
**Winner: Airline.**

### **3.2 Partition Control (`partitionBy`)**
- **Airline:** Contains clean, native `YEAR` and `MONTH` columns with zero missing values.
- **Taxi:** Requires deriving Year/Month from timestamps. Furthermore, the audit revealed out-of-bounds records (e.g., 2002, 2026). Without explicit filtering, partitioning this dataset would create tiny outlier directories, artificially triggering the **Small File Problem**.
**Winner: Airline.**

### **3.3 Dataset Scale & Local Execution**
- **Taxi (41M rows):** Excellent for heavy stress-testing, but introduces unnecessary CPU and I/O overhead for a local Spark environment.
- **Airline (5.8M rows):** Strikes the perfect balance. It is large enough to demonstrate clear I/O differences between row-based (CSV/JSON) and columnar (Parquet/ORC) formats, while remaining fast enough to support multiple benchmark iterations (warm-ups + measured runs).
**Winner: Airline.**

### **3.4 Experimental Fairness (Note on Source Formats)**
Airline originates as CSV, while Taxi originates as Parquet. Raw file sizes cannot be directly compared. To ensure fairness, the benchmarking script will read the chosen dataset, standardize its schema, `cache()` it in memory, and *then* execute the writes to CSV, JSON, Parquet, and ORC under identical conditions.

## **4. Final Dataset Selection & Project Lock**

Based on the audit, we officially select **Candidate A (Kaggle 2015 Flight Delays and Cancellations)** for Task 2. 

It provides an exact semantic match to the assignment requirements, possesses native partitioning columns, requires minimal preprocessing, and enables a highly reproducible local testing workflow.

### **Canonical Schema Lock**
From this point forward, the dataset is locked for the benchmarking pipeline. The schema will be standardized as follows to serve as the unified input for both ID3 (Write Benchmark) and ID4 (Read & Partition Experiments):

| Original Column | Canonical Column | Data Type | Purpose |
|---|---|---|---|
| `AIRLINE` | `Carrier` | String | Categorical Grouping (`GROUP BY`) |
| `ARRIVAL_DELAY` | `ArrDelay` | Integer | Numeric Aggregation (`AVG`) |
| `YEAR` | `Year` | Integer | Directory Partitioning |
| `MONTH` | `Month` | Integer | Directory Partitioning |

*Note: Any preprocessing (e.g., filtering, renaming) will be applied once during the DataFrame initialization phase. This protocol is now frozen to ensure consistency when handing off outputs to ID4 and ID6.*